# triangulation

The triangulation API reconstructs 3D landmarks from multiple camera measurements. `TriangulationParameters` configures robust validity checks, while `TriangulationResult` distinguishes a valid point from geometric failure modes.

GTSAM Copyright 2010-2026, Georgia Tech Research Corporation,
Atlanta, Georgia 30332-0415
All Rights Reserved

Authors: Frank Dellaert, et al. (see THANKS for the full author list)

See LICENSE for the license information

<a href="https://colab.research.google.com/github/borglab/gtsam/blob/develop/gtsam/geometry/doc/triangulation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
try:
    import google.colab
    %pip install --quiet gtsam-develop
except ImportError:
    pass

In [ ]:
import gtsam
import numpy as np

## A two-view track

Build two calibrated cameras, project a known point, and place the cameras in a matching `CameraSet`. Real applications supply measured pixels rather than generating them this way.

In [ ]:
K = gtsam.Cal3_S2(800.0, 800.0, 0.0, 320.0, 240.0)
camera0 = gtsam.PinholeCameraCal3_S2(gtsam.Pose3(), K)
camera1 = gtsam.PinholeCameraCal3_S2(
    gtsam.Pose3(gtsam.Rot3(), np.array([1.0, 0.0, 0.0])), K
)
point = np.array([0.2, -0.1, 5.0])
cameras = gtsam.CameraSetCal3_S2([camera0, camera1])
measurements = [camera.project(point) for camera in cameras]

## `TriangulationParameters`

The parameters control rank tolerance, EPI correction, maximum landmark distance, dynamic outlier rejection, the LOST algorithm, and an optional measurement noise model. Negative distance/outlier thresholds disable those checks.

In [ ]:
parameters = gtsam.TriangulationParameters(
    rankTolerance=1e-9,
    enableEPI=False,
    landmarkDistanceThreshold=10.0,
    dynamicOutlierRejectionThreshold=5.0,
    useLOST=False,
)
print("distance threshold:", parameters.landmarkDistanceThreshold)

## `TriangulationResult`

`triangulateSafe()` returns a result rather than throwing for common geometric failures. Test `valid()` before calling `get()`. The predicates `degenerate()`, `outlier()`, `farPoint()`, and `behindCamera()` explain invalid results.

In [ ]:
result = gtsam.triangulateSafe(cameras, measurements, parameters)
print("status:", result.status)
print("valid:", result.valid())
assert result.valid()
np.testing.assert_allclose(result.get(), point, atol=1e-8)

## Other triangulation entry points

`triangulatePoint3()` performs linear triangulation with optional nonlinear refinement, while `triangulateNonlinear()` refines an initial point. Overloads accept shared calibrations, camera sets with individual calibrations, spherical cameras, and batches of tracks.

## Source

[`triangulation.h`](../triangulation.h)

## AI assistance caveat

AI was used to help draft this documentation, and inaccuracies could be present.